# Four literature b-value claims tested on the six sub-regions

Percentages of units (grid nodes, GCMT mechanisms, M≥6 events, alarm episodes) that support each claim, with the paper's own
settings (N, Mc, dmc = 0.1, grid radius 0.5°, N per node). No hypothesis tests. Analysis functions live in `claims_lib.py`
(paper helpers copied verbatim from `bvalue4longmenshan.ipynb`), table and figures in `claims_report.py`.

Claims: **C1** low b ↔ high stress, **C2** high b ↔ low stress, **C3** high b ↔ fluids, **C4** b drops before (C4a) and rises
after (C4b) the marked events.

In [1]:
import os, time, warnings
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from IPython.display import Markdown, display
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
from claims_lib import *
from claims_report import *
OUT.mkdir(exist_ok=True)
figrun = load_figrun()
figrun[['region', 't_start', 't_end', 'n_events', 'mc', 'n_m', 'b_classical', 'b_classical_std', 'b_positive', 'b_positive_std']]

,region,t_start,t_end,n_events,mc,n_m,b_classical,b_classical_std,b_positive,b_positive_std
window,,,,,,,,,,
LMS_1970-2023,LMS,1970-01-13,2023-08-13,163566,1.9,1000,0.742,0.003,1.046,0.006
LMS_2006-2010,LMS,2006-01-01,2010-12-31,56287,1.9,500,0.761,0.005,0.987,0.009
LMS_2011-2016,LMS,2011-01-01,2016-12-31,55362,1.9,500,0.881,0.006,1.005,0.011
LMS_2016-2023,LMS,2016-01-01,2023-08-13,38226,1.9,100,0.948,0.008,1.054,0.014
XSH_1970-2023,XSH,1970-01-29,2023-08-13,42640,1.9,800,0.794,0.006,1.210,0.016
XSH_2008-2023,XSH,2008-01-01,2023-08-13,36167,1.9,800,1.045,0.010,1.168,0.019
XSH_2022-2023,XSH,2022-01-01,2023-08-13,18795,1.3,300,0.833,0.006,1.107,0.015
WYN_1970-2023,WYN,1970-01-01,2023-08-13,165944,1.9,1000,0.698,0.002,1.099,0.006
WYN_2000-2023,WYN,2000-01-01,2023-08-13,137121,1.9,1000,0.941,0.004,1.095,0.007


## 1. Validation: reproduce the paper's whole-window values and the SeismoStats sliding window

In [2]:
from seismostats.plots import plot_b_series_constant_nm
for label in ['LMS_2006-2010', 'XSH_2022-2023', 'NCP_2008-2023']:
    t0, t1 = window_span(label); df = load_region(figrun.loc[label, 'region'], t0, t1)
    mc = float(estimate_mc_maxc(df['mag'].to_numpy(), fmd_bin=0.1)[0])
    ec = b_window(df['mag'], df['time'], figrun.loc[label, 'mc'], ClassicBValueEstimator)
    ep = b_window(df['mag'], df['time'], figrun.loc[label, 'mc'], BPositiveBValueEstimator)
    assert len(df) == figrun.loc[label, 'n_events'] and mc == figrun.loc[label, 'mc']
    assert round(ec.b_value, 3) == figrun.loc[label, 'b_classical'] and round(ep.b_value, 3) == figrun.loc[label, 'b_positive']
    assert round(ec.std, 3) == figrun.loc[label, 'b_classical_std'] and round(ep.std, 3) == figrun.loc[label, 'b_positive_std']
    print(f'{label}: n={len(df)}, Mc={mc}, b_c={ec.b_value:.3f}±{ec.std:.3f}, b_p={ep.b_value:.3f}±{ep.std:.3f}  == figrun')
# sliding window vs seismostats.plots.plot_b_series_constant_nm (XSH 2022-2023, N=300)
label = 'XSH_2022-2023'; t0, t1 = window_span(label); df = load_region('XSH', t0, t1); mc = figrun.loc[label, 'mc']; n_m = int(figrun.loc[label, 'n_m'])
for cls in (ClassicBValueEstimator, BPositiveBValueEstimator):
    s = b_series(df['mag'], df['time'], mc, n_m, cls)
    fig, ax = plt.subplots(); kw = dict(dmc=0.1) if cls is BPositiveBValueEstimator else {}
    plot_b_series_constant_nm(df['mag'].to_numpy(), 0.1, mc, df['time'].to_numpy(), n_m=n_m, x_variable=df['time'].to_numpy(),
                              plot_technique='right', ax=ax, b_method=cls, **kw)
    y = ax.lines[0].get_ydata(); y = y[~np.isnan(y)]; plt.close(fig)
    assert np.allclose(y, s['b'].to_numpy()); print(cls.__name__, 'series reproduced,', len(s), 'windows')

LMS_2006-2010: n=56287, Mc=1.9, b_c=0.761±0.005, b_p=0.987±0.009  == figrun
XSH_2022-2023: n=18795, Mc=1.3, b_c=0.833±0.006, b_p=1.107±0.015  == figrun
NCP_2008-2023: n=53458, Mc=1.3, b_c=0.809±0.004, b_p=0.941±0.008  == figrun


ClassicBValueEstimator series reproduced, 11690 windows


BPositiveBValueEstimator series reproduced, 11690 windows


Sanity figure for the C4 rule on a constant-b synthetic catalogue (b = 1, Mc 1.9, magnitudes binned to 0.1, Poisson times,
N = 500): the share of windows ≥10% below the whole-period b (the alarm rule).

In [3]:
rng = np.random.default_rng(0)
n = 60000; mags = bin_to_precision(1.9 + rng.exponential(1 / np.log(10), n), 0.1)
times = (pd.Timestamp('2008-01-01') + pd.to_timedelta(np.sort(rng.uniform(0, 15 * 365.25, n)), unit='D')).to_numpy()
for cls in (ClassicBValueEstimator, BPositiveBValueEstimator):
    s = b_series(mags, times, 1.9, 500, cls); b_ref = b_window(mags, times, 1.9, cls).b_value
    print(f'{cls.__name__}: windows ≥10% below the whole-period b: {100 * ((s.b - b_ref) / b_ref <= -0.10).mean():.1f}%')

ClassicBValueEstimator: windows ≥10% below the whole-period b: 0.3%


BPositiveBValueEstimator: windows ≥10% below the whole-period b: 5.5%


## 2. Spatial b-maps 2008–2023, strain rate, heat flow (nodes_<REG>.csv)

Threshold b₀ = whole-window b of the same region, period and estimator (Mc = MAXC + 0.2 of the 2008–2023 catalogue).
Strain-rate tensor from Wang & Shen (2020) velocities (`ws_`), and from the SCB-fixed Sichuan–Yunnan field (`ds_`) for the
four SW regions. Node spacing 0.05°; N per node and the 0.5° radius as in the paper.

In [4]:
ws = load_gnss_ws(); ds = load_gnss_ds01(); hf = load_heatflow()
thr = {}
for region in REGIONS:
    t = time.time(); df = load_region(region, MAP_T0, MAP_T1)
    mc = float(estimate_mc_maxc(df['mag'].to_numpy(), fmd_bin=0.1)[0])
    bc = b_window(df['mag'], df['time'], mc, ClassicBValueEstimator); bp = b_window(df['mag'], df['time'], mc, BPositiveBValueEstimator)
    thr[region] = dict(mc=mc, n=len(df), b_c=bc.b_value, b_p=bp.b_value, n_c=bc.n, n_p=bp.n)   # n_c / n_p = sample sizes of the two regional fits (Utsu test)
    nc = bmap_nodes(df, mc, N_PER_GRID[region], ClassicBValueEstimator, BOXES[region])
    npos = bmap_nodes(df, mc, N_PER_GRID[region], BPositiveBValueEstimator, BOXES[region])
    nodes = nc.rename(columns={'b': 'b_c', 'std': 'std_c', 'n': 'n_c'}).drop(columns=['radius'])
    nodes['b_p'], nodes['std_p'], nodes['n_p'], nodes['radius'] = npos['b'], npos['std'], npos['n'], nc['radius']
    nodes = pd.concat([nodes, strain_at(nodes, ws).add_prefix('ws_')], axis=1)
    if region in ('LMS', 'XSH', 'WYN', 'XJ'):
        nodes = pd.concat([nodes, strain_at(nodes, ds).add_prefix('ds_')], axis=1)
    h = heatflow_at(nodes, hf); nodes['hf'], nodes['n_hf'] = h['hf'], h['n_hf']
    nodes['region'] = region
    nodes.to_csv(OUT / f'nodes_{region}.csv', index=False)
    v = nodes[np.isfinite(nodes.b_c)]
    print(f"{region}: Mc {mc}, n {len(df)}, b0 {bc.b_value:.3f} | {bp.b_value:.3f}; valid nodes {len(v)}, with strain {np.isfinite(v.ws_e2inv).sum()}, "
          f"median strain {np.nanmedian(v.ws_e2inv):.0f} nstrain/yr, heat-flow nodes {np.isfinite(v.hf).sum()}  ({time.time() - t:.0f}s)")
thr = pd.DataFrame(thr).T; thr.to_csv(OUT / 'thresholds_2008-2023.csv'); thr

LMS: Mc 1.9, n 143584, b0 0.826 | 1.003; valid nodes 4011, with strain 4011, median strain 12 nstrain/yr, heat-flow nodes 1407  (1s)


XSH: Mc 1.9, n 36167, b0 1.045 | 1.168; valid nodes 1103, with strain 1103, median strain 50 nstrain/yr, heat-flow nodes 0  (0s)


WYN: Mc 1.9, n 116124, b0 0.985 | 1.120; valid nodes 4749, with strain 4749, median strain 32 nstrain/yr, heat-flow nodes 1199  (2s)


XJ: Mc 1.9, n 24171, b0 0.993 | 1.160; valid nodes 1736, with strain 1736, median strain 37 nstrain/yr, heat-flow nodes 248  (1s)


NCP: Mc 1.3, n 53458, b0 0.809 | 0.941; valid nodes 8332, with strain 8332, median strain 6 nstrain/yr, heat-flow nodes 3376  (4s)


NW: Mc 1.9, n 160139, b0 0.709 | 0.874; valid nodes 19429, with strain 18499, median strain 16 nstrain/yr, heat-flow nodes 2339  (13s)


,mc,n,b_c,b_p,n_c,n_p
LMS,1.9,143584.0,0.825908,1.003216,47034.0,21029.0
XSH,1.9,36167.0,1.044514,1.168056,8736.0,3809.0
WYN,1.9,116124.0,0.984553,1.119767,36179.0,15809.0
XJ,1.9,24171.0,0.993381,1.160282,10800.0,4739.0
NCP,1.3,53458.0,0.808902,0.941018,26102.0,11817.0
NW,1.9,160139.0,0.709134,0.874127,97704.0,44456.0


## 3. C4 on the main-text windows (Figs 4–10): marked events, alarm episodes (events_C4.csv, alarms_C4.csv)

Unit = each main-text window with its own N and Mc (the paper's settings); targets = the events marked in panel (d)
(fifth-largest rule). Rules follow the published definitions (revised 2026-10-29):

- classical series: every window re-cut at max(panel Mc, MAXC + 0.2 of the window), ≥ 50 events above it
  (Gulia & Wiemer 2019, Methods steps 5–6a; Gulia et al. 2018); b-positive: panel Mc, dmc = 0.1, no re-cut and no
  exclusion (van der Elst 2021; Lippiello & Petrillo 2024).
- reference of a window = median of the earlier windows of the panel that share no event with it, leaving out windows
  that end inside the Gardner–Knopoff time window of an earlier marked event (Gulia & Wiemer 2019, 6a; Gulia et al. 2018);
  fewer than 5 eligible windows → the region's 2008–2023 whole-period b (Gulia & Wiemer 2019, 6b).
- drop = last window before the event ≥ 10% below its reference; rise = first post-event window ≥ 10% above the same
  reference (Gulia & Wiemer 2019, step 7). The classical post window starts when the per-window Mc is back at the panel
  floor (Gulia et al. 2018: events removed until Mc(t) matches the pre-cut level); the b-positive post window is the
  first window whose events all follow the event (van der Elst 2021).
- alarm episodes (runs of windows ≥ 10% below the reference) and their 7-day M≥4 follow-up are still written to
  alarms_C4.csv, but the false-alarm column was removed from the table on 2026-10-29: a false-alarm count needs an a priori
  target and validity window (Gulia et al. 2024 use a larger event within 3 months and the G-K radius, a question unrelated
  to this section), which regional windows cannot supply without arbitrary choices.

In [5]:
# the fast MAXC used for the per-window re-cut must equal seismostats' estimate_mc_maxc
rng = np.random.default_rng(0)
for label in MAIN_LABELS:
    t0, t1 = window_span(label); df_ = load_region(figrun.loc[label, 'region'], t0, t1); n_ = int(figrun.loc[label, 'n_m'])
    for i0 in rng.integers(0, max(1, len(df_) - n_), 20):
        w_ = df_['mag'].to_numpy()[i0:i0 + n_]
        assert np.isclose(maxc(w_), estimate_mc_maxc(w_, fmd_bin=0.1)[0]), (label, i0)
print('maxc == estimate_mc_maxc on 140 random windows')
EV, EP, SM = [], [], []
for label in MAIN_LABELS:
    for cls in (ClassicBValueEstimator, BPositiveBValueEstimator):
        ev, ep, sm = window_c4(label, cls, figrun); EV.append(ev); EP.append(ep); SM.append(sm)
events = pd.concat(EV, ignore_index=True); alarms = pd.concat(EP, ignore_index=True); c4_summary = pd.DataFrame(SM)
events.to_csv(OUT / 'events_C4.csv', index=False); alarms.to_csv(OUT / 'alarms_C4.csv', index=False); c4_summary.to_csv(OUT / 'alarms_C4_summary.csv', index=False)
c4_summary[['fig', 'window', 'estimator', 'n_m', 'mc', 'b_ww', 'b_regional', 'n_windows', 'n_windows_valid', 'n_windows_in_seq', 'n_marked', 'n_ref_regional',
            'drop_k', 'drop_n', 'rise_k', 'rise_n', 'n_episodes', 'n_hits', 'hit_ref_share']].round(3)

maxc == estimate_mc_maxc on 140 random windows


,fig,window,estimator,n_m,mc,b_ww,b_regional,n_windows,n_windows_valid,n_windows_in_seq,n_marked,n_ref_regional,drop_k,drop_n,rise_k,rise_n,n_episodes,n_hits,hit_ref_share
0,Fig. 4,LMS_2006-2010,classical,500,1.9,0.761,0.826,22580,22580,21988,9,0,5,9,0,9,98,67,0.358
1,Fig. 4,LMS_2006-2010,positive,500,1.9,0.987,1.003,22580,22580,21988,9,0,5,9,0,9,137,84,0.358
2,Fig. 5,LMS_2011-2016,classical,500,1.9,0.881,0.826,15146,15146,7142,5,0,3,5,0,5,23,11,0.240
3,Fig. 5,LMS_2011-2016,positive,500,1.9,1.005,1.003,15146,15146,7142,5,0,2,5,0,5,32,21,0.240
4,Fig. 6,XSH_2022-2023,classical,300,1.3,0.833,1.045,11690,11690,11585,6,1,2,6,1,6,87,21,0.102
5,Fig. 6,XSH_2022-2023,positive,300,1.3,1.107,1.168,11690,11690,11585,6,1,0,6,1,6,40,17,0.102
6,Fig. 7,WYN_2020-2023,classical,500,1.9,0.964,0.985,5204,5204,3570,7,0,4,7,0,7,7,2,0.086
7,Fig. 7,WYN_2020-2023,positive,500,1.9,1.055,1.120,5204,5204,3570,7,0,4,7,0,7,27,7,0.086
8,Fig. 8,XJ_2016-2023,classical,150,1.9,1.049,0.993,4784,4074,846,6,0,2,5,0,5,53,1,0.028
9,Fig. 8,XJ_2016-2023,positive,150,1.9,1.238,1.160,4784,4784,846,6,0,4,6,0,6,54,4,0.028


In [6]:
events[['fig', 'estimator', 'number', 'time', 'mag', 'initiator', 'ref', 'ref_source', 'n_ref', 'b_pre', 'mc_pre', 'pre_days', 'drop_rel', 'drop',
        'b_post', 'mc_post', 'post_start_days', 'post_days', 'rise_rel', 'rise']].round(3)

,fig,estimator,number,time,mag,initiator,ref,ref_source,n_ref,b_pre,mc_pre,pre_days,drop_rel,drop,b_post,mc_post,post_start_days,post_days,rise_rel,rise
0,Fig. 4,classical,1,2008-05-12 14:28:04.000,7.9,True,1.141,panel,91,1.174,2.3,372.394,-0.029,False,0.648,2.1,130.418,15.839,-0.432,False
1,Fig. 4,classical,2,2008-05-12 14:43:14.400,5.8,False,1.141,panel,94,1.030,2.3,369.547,0.097,False,0.648,2.1,130.407,15.839,-0.432,False
2,Fig. 4,classical,3,2008-05-12 19:11:00.100,5.8,False,1.149,panel,148,0.517,2.3,333.095,0.550,True,0.648,2.1,130.221,15.839,-0.436,False
3,Fig. 4,classical,4,2008-05-13 15:07:07.200,5.8,False,1.159,panel,286,0.306,2.2,245.860,0.736,True,0.648,2.1,129.391,15.839,-0.441,False
4,Fig. 4,classical,5,2008-05-18 01:08:24.800,5.8,False,1.215,panel,456,1.123,4.2,104.986,0.076,False,0.676,2.2,122.907,17.500,-0.444,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,Fig. 10,positive,2,2008-10-05 23:52:47.400,6.5,True,0.846,panel,9123,0.730,1.9,177.342,0.138,True,0.841,1.9,0.000,69.775,-0.007,False
84,Fig. 10,positive,3,2008-10-05 23:52:48.790,6.5,True,0.846,panel,9123,0.726,1.9,177.355,0.142,True,0.841,1.9,0.002,69.886,-0.007,False
85,Fig. 10,positive,4,2014-02-12 17:19:48.300,6.9,True,0.878,panel,22899,0.835,1.9,185.080,0.048,False,0.903,1.9,0.003,148.969,0.029,False
86,Fig. 10,positive,5,2016-06-26 19:17:09.600,6.5,True,0.873,panel,25900,0.808,1.9,328.676,0.074,False,0.829,1.9,0.006,177.047,-0.050,False


SI sensitivity: b-positive with dmc = 0.2 (van der Elst 2021's default; the paper's panel (c) checks 0.1/0.2/0.4).

In [7]:
SM2 = [window_c4(label, BPositiveBValueEstimator, figrun, dmc=0.2)[2] for label in MAIN_LABELS]
c4_dmc02 = pd.DataFrame(SM2); c4_dmc02.to_csv(OUT / 'alarms_C4_summary_dmc02.csv', index=False)
cmp = c4_summary[c4_summary.estimator == 'positive'].set_index('fig')[['drop_k', 'drop_n', 'rise_k', 'rise_n', 'n_episodes', 'n_hits']].join(
    c4_dmc02.set_index('fig')[['drop_k', 'drop_n', 'rise_k', 'rise_n', 'n_episodes', 'n_hits']], rsuffix='_dmc02')
cmp

,drop_k,drop_n,rise_k,rise_n,n_episodes,n_hits,drop_k_dmc02,drop_n_dmc02,rise_k_dmc02,rise_n_dmc02,n_episodes_dmc02,n_hits_dmc02
fig,,,,,,,,,,,,
Fig. 4,5,9,0,9,137,84,5,9,0,9,138,95
Fig. 5,2,5,0,5,32,21,2,5,0,5,48,31
Fig. 6,0,6,1,6,40,17,1,6,0,6,71,17
Fig. 7,4,7,0,7,27,7,6,7,0,7,14,6
Fig. 8,4,6,0,6,54,4,3,6,1,6,44,4
Fig. 9,1,5,0,5,36,5,2,5,1,5,48,1
Fig. 10,2,6,0,6,8,7,2,6,0,6,7,2


## 4. GCMT mechanisms and SI rows (epicentre b, depth)

In [8]:
gcmt = parse_gcmt()
lines = []
for region in REGIONS:
    g = gcmt[in_box(gcmt.lon, gcmt.lat, BOXES[region]) & (gcmt.mw >= 5.0)]
    vc = g['style'].value_counts()
    lines.append(dict(region=region, n=len(g), **{k: int(vc.get(k, 0)) for k in ('thrust', 'normal', 'strike-slip', 'oblique')},
                      dominant=vc.index[0] if len(vc) else '', dominant_pct=100 * vc.iloc[0] / len(g) if len(g) else np.nan))
gcmt_summary = pd.DataFrame(lines); gcmt_summary.to_csv(OUT / 'gcmt_summary.csv', index=False); gcmt_summary

,region,n,thrust,normal,strike-slip,oblique,dominant,dominant_pct
0,LMS,36,21,1,11,3,thrust,58.333333
1,XSH,8,0,3,5,0,strike-slip,62.500000
2,WYN,58,5,13,30,10,strike-slip,51.724138
3,XJ,11,0,0,8,3,strike-slip,72.727273
4,NCP,11,2,1,7,1,strike-slip,63.636364
5,NW,268,117,23,67,61,thrust,43.656716


In [9]:
EPI, DEP = [], []
for region in REGIONS:
    ev = targets(region, 6.0)
    if len(ev): EPI.append(epicentre_rows(region, ev, figrun))
    bins, summ = depth_rows(region, float(thr.loc[region, 'mc'])); DEP.append(bins)
    print(region, 'depth bins with b decreasing (classical | positive):', summ['all_dec_c'], '|', summ['all_dec_p'])
epi = pd.concat(EPI, ignore_index=True); epi.to_csv(OUT / 'epicentres_SI.csv', index=False)
pd.concat(DEP, ignore_index=True).to_csv(OUT / 'depth_SI.csv', index=False)
epi[[c for c in ['region', 'time', 'mag', 'mc', 'n', 'b0_c', 'b_epi_c', 'low_c', 'pct_c', 'b0_p', 'b_epi_p', 'low_p', 'pct_p', 'note'] if c in epi]].round(3)

LMS depth bins with b decreasing (classical | positive): 5/5 | 5/5


XSH depth bins with b decreasing (classical | positive): 3/4 | 3/4


WYN depth bins with b decreasing (classical | positive): 3/4 | 3/4


XJ depth bins with b decreasing (classical | positive): 3/4 | 3/4
NCP depth bins with b decreasing (classical | positive): 3/5 | 2/5


NW depth bins with b decreasing (classical | positive): 4/5 | 3/5


,region,time,mag,mc,n,b0_c,b_epi_c,low_c,pct_c,b0_p,b_epi_p,low_p,pct_p,note
0,LMS,2008-05-12 14:28:04.000,7.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,too few events
1,LMS,2008-05-25 16:21:49.300,6.1,4.2,794.0,1.139,NaN,None,NaN,1.461,NaN,None,NaN,NaN
2,LMS,2013-04-20 08:02:47.500,6.5,1.9,73710.0,0.787,0.798,False,0.359,0.992,1.025,False,0.437,NaN
3,LMS,2017-08-08 21:19:48.700,6.5,1.9,112959.0,0.803,0.805,False,0.317,0.992,0.930,True,0.229,NaN
4,XSH,2014-11-22 16:55:28.000,6.1,1.9,6363.0,1.066,0.885,True,0.094,1.180,0.966,True,0.140,NaN
5,XSH,2022-09-05 12:52:18.000,6.6,1.9,14405.0,1.091,1.076,True,0.397,1.191,1.137,True,0.287,NaN
6,WYN,2009-07-09 19:19:14.200,6.0,1.9,5697.0,0.840,NaN,None,NaN,1.020,NaN,None,NaN,NaN
7,WYN,2021-05-21 21:48:35.000,6.2,1.9,77484.0,1.018,0.778,True,0.010,1.153,0.891,True,0.037,NaN
8,XJ,2014-08-03 16:30:12.000,6.3,1.9,7094.0,0.911,1.036,False,0.728,1.048,1.215,False,0.637,NaN
9,NW,2008-03-21 06:33:00.100,6.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,too few events


## 5. Table and figures

In [10]:
tab = build_table(thr, gcmt); tab.to_csv(OUT / 'claims_table.csv', index=False)
c4 = build_c4_table(SM); c4.to_csv(OUT / 'claims_table_C4.csv', index=False)
mdtxt = table_markdown(tab, c4); open(OUT / 'claims_table.md', 'w').write(mdtxt); display(Markdown(mdtxt))
c4_long, c4_md = c4_event_table(events)   # Form B (chosen 2026-10-29): one row per window, one cell per marked event
c4_long.to_csv(OUT / 'table_C4_events.csv', index=False); open(OUT / 'table_C4_events.md', 'w').write(c4_md); display(Markdown(c4_md))

**Spatial claims** (cell = k/n (%), classical | positive; ref. = share expected with no relation)

| Claim | Row | LMS | XSH | WYN | XJ | NCP | NW |
|---|---|---|---|---|---|---|---|
| C1-C2 | b and strain rate on opposite sides (low-b & high rate, high-b & low rate) | 1405/4011 (35%) \| 1833/4011 (46%)<br>ref. 50% | 608/1103 (55%) \| 597/1103 (54%)<br>ref. 50% | 2406/4749 (51%) \| 2457/4749 (52%)<br>ref. 50% | 865/1736 (50%) \| 778/1736 (45%)<br>ref. 50% | 4318/8332 (52%) \| 3990/8332 (48%)<br>ref. 50% | 10626/18499 (57%) \| 11069/18499 (60%)<br>ref. 50% |
| C1 | Contractional nodes that are low-b | 254/1337 (19%) \| 479/1337 (36%)<br>ref. 34/36% | 94/368 (26%) \| 120/368 (33%)<br>ref. 36/41% | 483/1583 (31%) \| 602/1583 (38%)<br>ref. 35/45% | 216/579 (37%) \| 184/579 (32%)<br>ref. 41/35% | 550/2778 (20%) \| 494/2778 (18%)<br>ref. 30/25% | 1895/6166 (31%) \| 1540/6166 (25%)<br>ref. 33/21% |
| C2 | Extensional nodes that are high-b | 742/1337 (55%) \| 921/1337 (69%)<br>ref. 66/64% | 245/368 (67%) \| 223/368 (61%)<br>ref. 64/59% | 986/1583 (62%) \| 785/1583 (50%)<br>ref. 65/55% | 332/579 (57%) \| 339/579 (59%)<br>ref. 59/65% | 1650/2778 (59%) \| 1920/2778 (69%)<br>ref. 70/75% | 3096/6167 (50%) \| 4732/6167 (77%)<br>ref. 67/79% |
| C3 | High-heat-flow nodes that are high-b | 256/703 (36%) \| 375/703 (53%)<br>ref. 66/64% | n/a \| n/a | 455/599 (76%) \| 372/599 (62%)<br>ref. 65/55% | 112/124 (90%) \| 112/124 (90%)<br>ref. 59/65% | 1130/1688 (67%) \| 1398/1688 (83%)<br>ref. 70/75% | 494/1169 (42%) \| 907/1169 (78%)<br>ref. 67/79% |

**Temporal claims (C4), main-text windows** (cell = k/n (%), classical | positive)

| Window | Marked events | Marked events preceded by a b drop | Marked events followed by a b rise |
|---|---|---|---|
| Fig. 4 LMS_2006-2010 | 9 | 5/9 (56%) \| 5/9 (56%) | 0/9 (0%) \| 0/9 (0%) |
| Fig. 5 LMS_2011-2016 | 5 | 3/5 (60%) \| 2/5 (40%) | 0/5 (0%) \| 0/5 (0%) |
| Fig. 6 XSH_2022-2023 | 6 | 2/6 (33%) \| 0/6 (0%) | 1/6 (17%) \| 1/6 (17%) |
| Fig. 7 WYN_2020-2023 | 7 | 4/7 (57%) \| 4/7 (57%) | 0/7 (0%) \| 0/7 (0%) |
| Fig. 8 XJ_2016-2023 | 6 | 2/5 (40%) \| 4/6 (67%) | 0/5 (0%) \| 0/6 (0%) |
| Fig. 9 NCP_2008-2023 | 5 | 2/5 (40%) \| 1/5 (20%) | 0/5 (0%) \| 0/5 (0%) |
| Fig. 10 NW_2006-2023 | 6 | 3/6 (50%) \| 2/6 (33%) | 0/6 (0%) \| 0/6 (0%) |

Node classes low / high (% of valid nodes, classical | positive): LMS 34/66 \| 36/64; XSH 36/64 \| 41/59; WYN 35/65 \| 45/55; XJ 41/59 \| 35/65; NCP 30/70 \| 25/75; NW 33/67 \| 21/79.

Drop = last window before the event >= 10% below the pre-sequence reference (median of the earlier windows of the panel); rise = first post-event window >= 10% above the same reference; n = marked events with a valid window. Low-b / high-b node = below / above the regional whole-period b (b0; Schorlemmer et al. 2005, classes relative to the regional average). Restricting the classes to nodes that differ from b0 at Utsu (1992) p < 0.05 (Wiemer & Wyss 1997; Schorlemmer et al. 2004) leaves the direction of every cell unchanged (rows *_utsu in claims_table.csv). Strain-rate row: n = all nodes with strain data; chance = 50% because the strain split is at the node median.

**C4 per marked event, main-text windows** (cell = magnitude, * = aftershock of an earlier marked event, then classical | positive: ● drop before, ○ no drop, · no valid window)

| Window | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | Drop, c \| p |
|---|---|---|---|---|---|---|---|---|---|---|
| Fig. 4 LMS_2006-2010 | 7.9 ○○ | 5.8* ○○ | 5.8* ●○ | 5.8* ●○ | 5.8* ○● | 6.1* ○● | 5.8* ●● | 5.8* ●● | 5.8* ●● | 5/9 \| 5/9 |
| Fig. 5 LMS_2011-2016 | 6.5 ○○ | 5.2* ○○ | 5.2* ●○ | 5.2* ●● | 5.2* ●● |  |  |  |  | 3/5 \| 2/5 |
| Fig. 6 XSH_2022-2023 | 4.7 ○○ | 6.6 ○○ | 4.7* ●○ | 5.1* ○○ | 5.3* ○○ | 5.0* ●○ |  |  |  | 2/6 \| 0/6 |
| Fig. 7 WYN_2020-2023 | 4.6 ○○ | 5.4 ○○ | 4.6* ○○ | 6.2 ●● | 4.9* ●● | 5.1* ●● | 4.6* ●● |  |  | 4/7 \| 4/7 |
| Fig. 8 XJ_2016-2023 | 4.5 ○○ | 5.0 ●● | 4.9 ●● | 4.5 ○● | 4.4 ○● | 4.4 ·○ |  |  |  | 2/5 \| 4/6 |
| Fig. 9 NCP_2008-2023 | 4.7 ○○ | 4.5 ●○ | 4.5 ●● | 4.5 ○○ | 4.6 ○○ |  |  |  |  | 2/5 \| 1/5 |
| Fig. 10 NW_2006-2023 | 6.9 ○○ | 6.5 ●● | 6.5 ●● | 6.9 ○○ | 6.5 ●○ | 7.2 ○○ |  |  |  | 3/6 \| 2/6 |

Rise after (first post-event window >= 10% above the same reference), classical | positive: Fig. 4 0/9 | 0/9; Fig. 5 0/5 | 0/5; Fig. 6 1/6 | 1/6; Fig. 7 0/7 | 0/7; Fig. 8 0/5 | 0/6; Fig. 9 0/5 | 0/5; Fig. 10 0/6 | 0/6.
Drop = last window ending before the event >= 10% below the pre-sequence reference (median of the earlier windows of the panel outside earlier sequences); events in time order within each window.

In [11]:
fig_summary(tab, c4, OUT / 'fig_summary.pdf', events=events)
fig_table(tab, events, OUT / 'table_main.pdf')   # main-text table (spatial rows + C4 Form B) rendered as one figure, 2026-10-30
figS_C12(thr, gcmt, OUT / 'figS_C12.pdf')
figS_C3(thr, hf, OUT / 'figS_C3.pdf')
figS_C4(events, figrun, OUT / 'figS_C4.pdf')
sorted(p.name for p in OUT.iterdir())

['alarms_C4.csv',
 'alarms_C4_summary.csv',
 'alarms_C4_summary_dmc02.csv',
 'claims_table.csv',
 'claims_table.md',
 'claims_table_C4.csv',
 'depth_SI.csv',
 'epicentres_SI.csv',
 'events_C4.csv',
 'figS_C12.pdf',
 'figS_C3.pdf',
 'figS_C4.pdf',
 'fig_summary.pdf',
 'gcmt_summary.csv',
 'nodes_LMS.csv',
 'nodes_NCP.csv',
 'nodes_NW.csv',
 'nodes_WYN.csv',
 'nodes_XJ.csv',
 'nodes_XSH.csv',
 'table_C4_events.csv',
 'table_C4_events.md',
 'table_main.pdf',
 'thresholds_2008-2023.csv']